Imports

In [23]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import numpy as np
from sklearn.metrics import accuracy_score

Load model and reevaluate on clean and master dataset

In [79]:
model = tf.keras.models.load_model("baseline_resnet152v2.keras")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 7, 7, 2048)     │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,056,325 (259.61 MB)

 Trainable params: 4,731,137 (18.05 MB)

 Non-trainable params: 53,862,912 (205.47 MB)

 Optimizer params: 9,462,276 (36.10 MB)

In [ ]:
model = tf.keras.models.load_model("baseline_resnet152v2.keras")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 7, 7, 2048)     │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,056,325 (259.61 MB)

 Trainable params: 4,731,137 (18.05 MB)

 Non-trainable params: 53,862,912 (205.47 MB)

 Optimizer params: 9,462,276 (36.10 MB)

FGSM Attack and Evaluate Functions

In [2]:
loss_fn = tf.keras.losses.BinaryCrossentropy()

def fgsm_attack(model, images, labels, epsilon=0.01):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.float32)
    labels = tf.reshape(labels, (-1, 1))  # match (batch,1)

    with tf.GradientTape() as tape:
        tape.watch(images)
        predictions = model(images, training=False)
        loss = loss_fn(labels, predictions)

    gradients = tape.gradient(loss, images)
    adv_images = images + epsilon * tf.sign(gradients)
    adv_images = tf.clip_by_value(adv_images, 0.0, 1.0)

    return adv_images

In [3]:
def evaluate_clean(model, generator):
    all_labels = []
    all_preds = []
    all_probs = []

    generator.reset()

    for i in range(len(generator)):
        images, labels = generator[i]

        probs = model.predict(images, verbose=0).flatten()
        preds = (probs > 0.5).astype(int)

        all_labels.extend(labels.astype(int))
        all_probs.extend(probs)
        all_preds.extend(preds)

    acc = accuracy_score(all_labels, all_preds)

    return np.array(all_labels), np.array(all_preds), np.array(all_probs), acc

In [4]:
def evaluate_fgsm(model, generator, epsilon=0.01):
    all_labels = []
    all_preds = []
    all_probs = []

    generator.reset()

    for i in range(len(generator)):
        images, labels = generator[i]

        images_tf = tf.convert_to_tensor(images, dtype=tf.float32)
        labels_tf = tf.convert_to_tensor(labels, dtype=tf.float32)

        adv_images = fgsm_attack(model, images_tf, labels_tf, epsilon=epsilon)

        probs = model.predict(adv_images, verbose=0).flatten()
        preds = (probs > 0.5).astype(int)

        all_labels.extend(labels.astype(int))
        all_probs.extend(probs)
        all_preds.extend(preds)

    acc = accuracy_score(all_labels, all_preds)

    return np.array(all_labels), np.array(all_preds), np.array(all_probs), acc

Evaluate on their clean dataset

In [83]:
IMG_SIZE = 224
BATCH = 16

datagen = ImageDataGenerator(rescale=1/255.)

Clean dataset (theirs)

In [84]:
ds_clean = datagen.flow_from_directory(
    r"chest_xray/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

print("Clean classes:", ds_clean.class_indices)

Found 624 images belonging to 2 classes.
Clean classes: {'NORMAL': 0, 'PNEUMONIA': 1}


Master Dataset

In [85]:
ds_master = datagen.flow_from_directory(
    r"Master_Dataset/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

print("Master classes:", ds_master.class_indices)

Found 1757 images belonging to 2 classes.
Master classes: {'NORMAL': 0, 'PNEUMONIA': 1}


Kermany Dataset

In [86]:
ds_kermany = datagen.flow_from_directory(
    r"Kermany_Pediatric_Attack",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

print("Kermany classes:", ds_master.class_indices)

Found 468 images belonging to 2 classes.
Kermany classes: {'NORMAL': 0, 'PNEUMONIA': 1}


Evaluate both theirs and master

In [89]:
labels_clean, preds_clean, _, clean_acc = evaluate_clean(model, ds_clean)
labels_master, preds_master, _, master_acc = evaluate_clean(model, ds_master)
labels_master, preds_master, _, kermany_acc = evaluate_clean(model, ds_kermany)


print("Clean (chest_xray):", clean_acc)
print("Clean (master):", master_acc)
print("Clean (kermany):", kermany_acc)

Clean (chest_xray): 0.9214743589743589
Clean (master): 0.6624928856004553
Clean (kermany): 0.8547008547008547


Evaluate on fgsm on both

In [ ]:
eps = 0.01

_, _, _, fgsm_clean_acc = evaluate_fgsm(model, ds_clean, epsilon=eps)
_, _, _, fgsm_master_acc = evaluate_fgsm(model, ds_master, epsilon=eps)
_, _, _, fgsm_kermany_acc = evaluate_fgsm(model, ds_kermany, epsilon=eps)

print("FGSM (chest_xray):", fgsm_clean_acc)
print("FGSM (master):", fgsm_master_acc)
print("FGSM (kermany):", fgsm_kermany_acc)

FGSM (chest_xray): 0.1987179487179487
FGSM (master): 0.12407512805919181
FGSM (master): 0.14957264957264957


Adversial Training

In [40]:
def adversarial_train_step(model, optimizer, images, labels, epsilon=0.01):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.float32)
    labels = tf.reshape(labels, (-1, 1))

    adv_images = fgsm_attack(model, images, labels, epsilon=epsilon)

    combined_images = tf.concat([images, adv_images], axis=0)
    combined_labels = tf.concat([labels, labels], axis=0)

    with tf.GradientTape() as tape:
        predictions = model(combined_images, training=True)
        loss = loss_fn(combined_labels, predictions)

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss

In [39]:
def adversarial_train(model, train_generator, val_generator, epochs=5, epsilon=0.01, lr=1e-4):
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        train_generator.reset()

        train_losses = []

        for i in range(len(train_generator)):
            images, labels = train_generator[i]
            loss = adversarial_train_step(model, optimizer, images, labels, epsilon=epsilon)
            train_losses.append(float(loss))

        print(f"Train loss: {np.mean(train_losses):.4f}")

        _, _, _, clean_acc = evaluate_clean(model, val_generator)
        _, _, _, fgsm_acc = evaluate_fgsm(model, val_generator, epsilon=epsilon)

        print(f"Val clean acc: {clean_acc:.4f}")
        print(f"Val FGSM acc:  {fgsm_acc:.4f}")

Datasets for adv

In [33]:
IMG_SIZE = 224
BATCH = 16
SEED = 42

train_val_datagen = ImageDataGenerator(
    rescale=1/255.,
    validation_split=0.2
)

ds_train = train_val_datagen.flow_from_directory(
    r"chest_xray/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=True,
    subset="training",
    seed=SEED
)

ds_val = train_val_datagen.flow_from_directory(
    r"chest_xray/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False,
    subset="validation",
    seed=SEED
)

test_datagen = ImageDataGenerator(rescale=1/255.)

ds_test = test_datagen.flow_from_directory(
    r"chest_xray/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [54]:
model = tf.keras.models.load_model("baseline_resnet152v2.keras")

adversarial_train(
    model,
    ds_train,
    ds_val,
    epochs=3,
    epsilon=0.01,
    lr=1e-5
)


Epoch 1/3
Train loss: 0.5288
Val clean acc: 0.9608
Val FGSM acc:  0.6794

Epoch 2/3
Train loss: 0.3900
Val clean acc: 0.9636
Val FGSM acc:  0.6947

Epoch 3/3
Train loss: 0.3397
Val clean acc: 0.9675
Val FGSM acc:  0.6976


In [55]:
model.save("baseline_resnet152v2_adversarial.keras")

In [34]:
model = tf.keras.models.load_model("baseline_resnet152v2_adversarial_finetunedonmaster.keras")

loss, acc = model.evaluate(ds_test, verbose=1)
print("Test Clean Loss:", loss)
print("Test Clean Accuracy:", acc)

39/39 ━━━━━━━━━━━━━━━━━━━━ 51s 943ms/step - accuracy: 0.8638 - loss: 0.3493
Test Clean Loss: 0.3492603898048401
Test Clean Accuracy: 0.8637820482254028


In [35]:
labels, preds, probs, fgsm_acc = evaluate_fgsm(model, ds_test)

print("Test FGSM Accuracy:", fgsm_acc)

Test FGSM Accuracy: 0.16185897435897437


==================================================================

Adversaraial Training on Master

In [17]:
model = tf.keras.models.load_model("baseline_resnet152v2_adversarial.keras")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 7, 7, 2048)     │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,056,325 (259.61 MB)

 Trainable params: 4,731,137 (18.05 MB)

 Non-trainable params: 53,862,912 (205.47 MB)

 Optimizer params: 9,462,276 (36.10 MB)

Dataset for Master

In [18]:
IMG_SIZE = 224
BATCH = 16
SEED = 42

train_val_datagen = ImageDataGenerator(
    rescale=1/255.,
    validation_split=0.2
)

ds_master_train = train_val_datagen.flow_from_directory(
    r"Master_Dataset/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=True,
    subset="training",
    seed=SEED
)

ds_master_val = train_val_datagen.flow_from_directory(
    r"Master_Dataset/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False,
    subset="validation",
    seed=SEED
)

test_datagen = ImageDataGenerator(rescale=1/255.)

ds_master_test = test_datagen.flow_from_directory(
    r"Master_Dataset/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

Found 11244 images belonging to 2 classes.
Found 350 images belonging to 2 classes.
Found 1757 images belonging to 2 classes.


Training

In [19]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=loss_fn,
    metrics=["accuracy"]
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    ds_master_train,
    validation_data=ds_master_val,
    epochs=15,
)

Epoch 1/15
703/703 ━━━━━━━━━━━━━━━━━━━━ 456s 636ms/step - accuracy: 0.7118 - loss: 0.5746 - val_accuracy: 0.8571 - val_loss: 0.3272
Epoch 2/15
703/703 ━━━━━━━━━━━━━━━━━━━━ 448s 638ms/step - accuracy: 0.7598 - loss: 0.4998 - val_accuracy: 0.8686 - val_loss: 0.3120
Epoch 3/15
703/703 ━━━━━━━━━━━━━━━━━━━━ 433s 616ms/step - accuracy: 0.7871 - loss: 0.4551 - val_accuracy: 0.8771 - val_loss: 0.3154
Epoch 4/15
703/703 ━━━━━━━━━━━━━━━━━━━━ 398s 565ms/step - accuracy: 0.8195 - loss: 0.4041 - val_accuracy: 0.8600 - val_loss: 0.3349
Epoch 5/15
703/703 ━━━━━━━━━━━━━━━━━━━━ 736s 1s/step - accuracy: 0.8531 - loss: 0.3474 - val_accuracy: 0.8771 - val_loss: 0.3009
Epoch 6/15
703/703 ━━━━━━━━━━━━━━━━━━━━ 935s 1s/step - accuracy: 0.8936 - loss: 0.2852 - val_accuracy: 0.8457 - val_loss: 0.3660
Epoch 7/15
703/703 ━━━━━━━━━━━━━━━━━━━━ 946s 1s/step - accuracy: 0.9267 - loss: 0.2247 - val_accuracy: 0.8457 - val_loss: 0.3722
Epoch 8/15
703/703 ━━━━━━━━━━━━━━━━━━━━ 783s 1s/step - accuracy: 0.9509 - loss: 0.171

In [20]:
loss, acc = model.evaluate(ds_master_test, verbose=1)
print("Test Loss:", loss)
print("Test Accuracy:", acc)

110/110 ━━━━━━━━━━━━━━━━━━━━ 99s 901ms/step - accuracy: 0.7484 - loss: 1.0218
Test Loss: 1.021767020225525
Test Accuracy: 0.748434841632843


In [21]:
model.save("baseline_resnet152v2_adversarial_finetunedonmaster15epochs.keras")

Final evaluation after adversarial Training

In [25]:
IMG_SIZE = 224
BATCH = 16

datagen = ImageDataGenerator(rescale=1/255.)

ds_clean = datagen.flow_from_directory(
    r"chest_xray/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

print("Clean classes:", ds_clean.class_indices)

ds_master = datagen.flow_from_directory(
    r"Master_Dataset/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

print("Master classes:", ds_master.class_indices)

ds_kermany = datagen.flow_from_directory(
    r"Kermany_Pediatric_Attack",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

print("Kermany classes:", ds_master.class_indices)


Found 624 images belonging to 2 classes.
Clean classes: {'NORMAL': 0, 'PNEUMONIA': 1}
Found 1757 images belonging to 2 classes.
Master classes: {'NORMAL': 0, 'PNEUMONIA': 1}
Found 468 images belonging to 2 classes.
Kermany classes: {'NORMAL': 0, 'PNEUMONIA': 1}


Evaluation on theirs, master and kermany on final model

In [29]:
model = tf.keras.models.load_model("baseline_resnet152v2_adversarial_finetunedonmaster.keras")

labels_clean, preds_clean, _, clean_acc = evaluate_clean(model, ds_clean)
labels_master, preds_master, _, master_acc = evaluate_clean(model, ds_master)
labels_master, preds_master, _, kermany_acc = evaluate_clean(model, ds_kermany)


print("Clean (chest_xray):", clean_acc)
print("Clean (master):", master_acc)
print("Clean (kermany):", kermany_acc)

Clean (chest_xray): 0.8637820512820513
Clean (master): 0.7734775184974388
Clean (kermany): 0.8547008547008547


Evaluation on theirs, master and kermany on baseline model

In [28]:
model = tf.keras.models.load_model("baseline_resnet152v2.keras")

labels_clean, preds_clean, _, clean_acc = evaluate_clean(model, ds_clean)
labels_master, preds_master, _, master_acc = evaluate_clean(model, ds_master)
labels_master, preds_master, _, kermany_acc = evaluate_clean(model, ds_kermany)


print("Clean (chest_xray):", clean_acc)
print("Clean (master):", master_acc)
print("Clean (kermany):", kermany_acc)

Clean (chest_xray): 0.9214743589743589
Clean (master): 0.6624928856004553
Clean (kermany): 0.8547008547008547


FINAL MODEL FGSM

In [30]:
model = tf.keras.models.load_model("baseline_resnet152v2_adversarial_finetunedonmaster.keras")
eps = 0.01

_, _, _, fgsm_clean_acc = evaluate_fgsm(model, ds_clean, epsilon=eps)
_, _, _, fgsm_master_acc = evaluate_fgsm(model, ds_master, epsilon=eps)
_, _, _, fgsm_kermany_acc = evaluate_fgsm(model, ds_kermany, epsilon=eps)

print("FGSM (chest_xray):", fgsm_clean_acc)
print("FGSM (master):", fgsm_master_acc)
print("FGSM (kermany):", fgsm_kermany_acc)

FGSM (chest_xray): 0.16185897435897437
FGSM (master): 0.13204325554923166
FGSM (kermany): 0.14743589743589744


BASELINE MODEL FGSM

In [31]:
model = tf.keras.models.load_model("baseline_resnet152v2.keras")
eps = 0.01

_, _, _, fgsm_clean_acc = evaluate_fgsm(model, ds_clean, epsilon=eps)
_, _, _, fgsm_master_acc = evaluate_fgsm(model, ds_master, epsilon=eps)
_, _, _, fgsm_kermany_acc = evaluate_fgsm(model, ds_kermany, epsilon=eps)

print("FGSM (chest_xray):", fgsm_clean_acc)
print("FGSM (master):", fgsm_master_acc)
print("FGSM (kermany):", fgsm_kermany_acc)

FGSM (chest_xray): 0.1987179487179487
FGSM (master): 0.12407512805919181
FGSM (kermany): 0.14957264957264957


===================================================================

comhbine master and theirs and fgsm train

In [37]:
IMG_SIZE = 224
BATCH = 16
SEED = 42

train_val_datagen = ImageDataGenerator(
    rescale=1/255.,
    validation_split=0.2
)

ds_train = train_val_datagen.flow_from_directory(
    r"combinedmasterandchestxray/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=True,
    subset="training",
    seed=SEED
)

ds_val = train_val_datagen.flow_from_directory(
    r"combinedmasterandchestxray/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False,
    subset="validation",
    seed=SEED
)

test_datagen = ImageDataGenerator(rescale=1/255.)

ds_test = test_datagen.flow_from_directory(
    r"combinedmasterandchestxray/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

Found 15429 images belonging to 2 classes.
Found 3857 images belonging to 2 classes.
Found 2381 images belonging to 2 classes.


In [41]:
model = tf.keras.models.load_model("baseline_resnet152v2.keras")

adversarial_train(
    model,
    ds_train,
    ds_val,
    epochs=3,
    epsilon=0.01,
    lr=1e-5
)


Epoch 1/3


KeyboardInterrupt: 